### Question 1

In [ ]:
graph = {
    'A': [('B', 1), ('C', 2)],
    'B': [('A', 1), ('D', 5), ('E', 2)],
    'C': [('A', 2), ('F', 3)],
    'D': [('B', 5), ('G', 4)],
    'E': [('B', 2), ('G', 1)],
    'F': [('C', 3), ('G', 1)],
    'G': []
}

heuristic = {
    'A': 6,
    'B': 4,
    'C': 2,
    'D': 3,
    'E': 1,
    'F': 1, # Added heuristic for 'F'
    'G': 0
}

print("Sample Graph (adjacency list):", graph)
print("Heuristic values:", heuristic)

Sample Graph (adjacency list): {'A': [('B', 1), ('C', 2)], 'B': [('A', 1), ('D', 5), ('E', 2)], 'C': [('A', 2), ('F', 3)], 'D': [('B', 5), ('G', 4)], 'E': [('B', 2), ('G', 1)], 'F': [('C', 3), ('G', 1)], 'G': []}
Heuristic values: {'A': 6, 'B': 4, 'C': 2, 'D': 3, 'E': 1, 'F': 1, 'G': 0}


In [ ]:
import heapq

# Greedy Best First Search
def GBFS(graph, heuristic, start, end):
    visited = []
    queue = []

    # Push the starting node into the priority queue
    heapq.heappush(queue, (heuristic[start], start))

    path_found = False
    path_trace = {}

    while queue:

        h_val, current_node = heapq.heappop(queue)

        if current_node == end:
            path_found = True
            break

        if current_node not in visited:
            visited.append(current_node)

            for child, _ in graph[current_node]:
                if child not in visited:
                    heapq.heappush(queue, (heuristic[child], child))
                    path_trace[child] = current_node
    if path_found:
        # Reconstruct path
        current = end
        path = [current]
        while current != start:
            current = path_trace[current]
            path.append(current)
        return path[::-1] # Reverse to get path from start to end
    else:
        return ["No Path Found!"]

In [ ]:
path = GBFS(graph, heuristic, 'A', 'G')
print("Path:", path)

Path: ['A', 'C', 'F', 'G']


### Question 2

In [ ]:
import heapq

# A Star Search
def AStar(graph, heuristic, start, end):
    # actual cost from start to current node
    cost = {node: float('inf') for node in graph} # Initialize all costs to infinity
    cost[start] = 0

    # estimated total cost from start to end through current node (cost + heuristic)
    total = {node: float('inf') for node in graph}
    total[start] = heuristic[start]

    queue = [(total[start], start)]

    path_trace = {}

    while queue:
        current_cost, current_node = heapq.heappop(queue)

        if current_node == end:
            # Reconstruct path
            path = [current_node]
            while current_node in path_trace:
                current_node = path_trace[current_node]
                path.append(current_node)
            return path[::-1], cost[end]

        if current_cost > total[current_node]:
            continue

        for child, weight in graph[current_node]:
            total_cost = cost[current_node] + weight

            if total_cost < cost[child]:
                # This path to neighbor is better than any previous one
                path_trace[child] = current_node
                cost[child] = total_cost
                total[child] = total_cost + heuristic[child]
                heapq.heappush(queue, (total[child], child))

    return "No Path Found!", "Null"

In [ ]:
path, cost = AStar(graph, heuristic, 'A', 'G')
print("Path:", path)
print("Cost:", cost)

Path: ['A', 'B', 'E', 'G']
Cost: 4


### Question 3

In [ ]:
import collections

# Using a tuple of tuples for immutability
initial_board = (
    (1, 2, 3),
    (4, 5, 6),
    (0, 7, 8)
)

goal_state = (
    (1, 2, 3),
    (4, 5, 6),
    (7, 8, 0)
)

In [ ]:
# helper functions

def find_blank(board):
    # Finds the (row, col) coordinates of the blank tile (0) on the board.
    for r_idx, row in enumerate(board):
        for c_idx, tile in enumerate(row):
            if tile == 0:
                return r_idx, c_idx
    return None

def get_neighbors(board):
    # Generates all possible next board states by moving the blank tile.
    neighbors = []
    blank_r, blank_c = find_blank(board)

    moves = [(-1, 0), (1, 0), (0, -1), (0, 1)] # Up, Down, Left, Right

    for dr, dc in moves:
        new_r, new_c = blank_r + dr, blank_c + dc

        # If the new position is within board boundaries
        if 0 <= new_r < 3 and 0 <= new_c < 3:
            # New board state by swapping blank with the tile at (new_r, new_c)
            new_board_list = [list(row) for row in board]

            # Swap the tiles
            new_board_list[blank_r][blank_c] = new_board_list[new_r][new_c]
            new_board_list[new_r][new_c] = 0

            # Convert back to tuple of tuples for immutability
            new_board_tuple = tuple(tuple(row) for row in new_board_list)
            neighbors.append(new_board_tuple)

    return neighbors

In [ ]:
print("Initial 8-puzzle board")
for row in initial_board:
    print(row)

print("\nGoal 8-puzzle board:")
for row in goal_state:
    print(row)

print("\nBlank tile coordinates for initial_board:", find_blank(initial_board))
print("\nNeighbors of initial_board:")
for i, neighbor in enumerate(get_neighbors(initial_board)):
  print(f"\nNeighbor {i+1}:")
  for row in neighbor:
    print(row)


Initial 8-puzzle board
(1, 2, 3)
(4, 5, 6)
(0, 7, 8)

Goal 8-puzzle board:
(1, 2, 3)
(4, 5, 6)
(7, 8, 0)

Blank tile coordinates for initial_board: (2, 0)

Neighbors of initial_board:

Neighbor 1:
(1, 2, 3)
(0, 5, 6)
(4, 7, 8)

Neighbor 2:
(1, 2, 3)
(4, 5, 6)
(7, 0, 8)


In [ ]:
def misplaced_tiles_heuristic(board, goal_state):
    misplaced_count = 0
    for r in range(3):
        for c in range(3):
            # Check if the tile is not the blank tile (0) and it's not in its goal position
            if board[r][c] != 0 and board[r][c] != goal_state[r][c]:
                misplaced_count += 1
    return misplaced_count

print("Misplaced Tiles Heuristic for initial_board:", misplaced_tiles_heuristic(initial_board, goal_state))

Misplaced Tiles Heuristic for initial_board: 2


In [ ]:
def manhattan_distance_heuristic(board, goal_state):
    distance = 0
    # Pre-compute goal positions for faster lookup
    goal_positions = {}
    for r_goal in range(3):
        for c_goal in range(3):
            tile = goal_state[r_goal][c_goal]
            goal_positions[tile] = (r_goal, c_goal)

    for r_curr in range(3):
        for c_curr in range(3):
            tile = board[r_curr][c_curr]
            if tile != 0: # Exclude the blank tile
                r_goal, c_goal = goal_positions[tile]
                distance += abs(r_curr - r_goal) + abs(c_curr - c_goal)
    return distance

print("Manhattan Distance Heuristic for initial_board:", manhattan_distance_heuristic(initial_board, goal_state))

Manhattan Distance Heuristic for initial_board: 2


In [ ]:
import heapq
import collections

def astar_search(initial_board, goal_state, heuristic_function):
    queue = [] # Priority queue of (f_score, board_state)
    heapq.heappush(queue, (heuristic_function(initial_board, goal_state), initial_board))

    came_from = {} # To reconstruct the path: child_board -> parent_board

    cost = collections.defaultdict(lambda: float('inf')) # Cost from start to current node
    cost[initial_board] = 0

    total = collections.defaultdict(lambda: float('inf')) # cost + heuristic
    total[initial_board] = heuristic_function(initial_board, goal_state)

    explored_states_count = 0

    while queue:
        current_total, current_board = heapq.heappop(queue)

        # If we have already found a shorter path to current_board, skip this one
        if current_total > total[current_board]:
            continue

        explored_states_count += 1

        if current_board == goal_state:
            path = []
            current = current_board
            while current in came_from:
                path.append(current)
                current = came_from[current]
            path.append(initial_board)
            return path[::-1], cost[goal_state], explored_states_count

        for neighbor_board in get_neighbors(current_board):
            # Cost of moving from current_board to neighbor is 1
            new_cost = cost[current_board] + 1

            if new_cost < cost[neighbor_board]:
                came_from[neighbor_board] = current_board
                cost[neighbor_board] = new_cost
                total[neighbor_board] = new_cost + heuristic_function(neighbor_board, goal_state)
                heapq.heappush(queue, (total[neighbor_board], neighbor_board))

    return "No Path Found!", None, explored_states_count

In [ ]:
# Test with Misplaced Tiles Heuristic
print("--- A* with Misplaced Tiles Heuristic ---")
path_mt, cost_mt, states_mt = astar_search(initial_board, goal_state, misplaced_tiles_heuristic)
print("\nPath:")
for i, row in enumerate(path_mt):
    print("\n Move {}" .format(i+1))
    for r in row:
        print(r)
print("\nCost:", cost_mt)
print("States Explored:", states_mt)

# Test with Manhattan Distance Heuristic
print("\n--- A* with Manhattan Distance Heuristic ---")
path_md, cost_md, states_md = astar_search(initial_board, goal_state, manhattan_distance_heuristic)
print("\nPath:")
for i, row in enumerate(path_md):
    print("\n Move {}" .format(i+1))
    for r in row:
        print(r)
print("\nCost:", cost_md)
print("States Explored:", states_md)

--- A* with Misplaced Tiles Heuristic ---

Path:

 Move 1
(1, 2, 3)
(4, 5, 6)
(0, 7, 8)

 Move 2
(1, 2, 3)
(4, 5, 6)
(7, 0, 8)

 Move 3
(1, 2, 3)
(4, 5, 6)
(7, 8, 0)

Cost: 2
States Explored: 3

--- A* with Manhattan Distance Heuristic ---

Path:

 Move 1
(1, 2, 3)
(4, 5, 6)
(0, 7, 8)

 Move 2
(1, 2, 3)
(4, 5, 6)
(7, 0, 8)

 Move 3
(1, 2, 3)
(4, 5, 6)
(7, 8, 0)

Cost: 2
States Explored: 3


### Question 4

In [16]:
import copy

# tic tac toe board states

empty_state = [
    [' ', ' ', ' '],
    [' ', ' ', ' '],
    [' ', ' ', ' ']
]

In [17]:
def possible_moves(current_state, turn):
  # turn either X or O
  possible_moves_list = []
  for i in range(3):
    for j in range(3):
      if current_state[i][j] == ' ':
        new_state = copy.deepcopy(current_state)
        new_state[i][j] = turn
        possible_moves_list.append(new_state)
  return possible_moves_list

In [18]:
# helping functions

def check_win(board, player):
    # Check rows
    for row in board:
        if all(s == player for s in row):
            return True

    # Check columns
    for col in range(3):
        if all(board[row][col] == player for row in range(3)):
            return True

    # Check diagonals
    if (board[0][0] == player and board[1][1] == player and board[2][2] == player) or \
       (board[0][2] == player and board[1][1] == player and board[2][0] == player):
        return True

    return False

def check_draw(board):
    # If there's a winner, it's not a draw
    if check_win(board, 'X') or check_win(board, 'O'):
        return False

    # Check if there are any empty cells
    for row in board:
        if ' ' in row:
            return False # Still moves possible

    return True # No winner and no empty cells

In [19]:
def get_utility(board, player_ai, player_opponent):
    # If AI wins
    if check_win(board, player_ai):
        return 1
    # If opponent wins
    if check_win(board, player_opponent):
        return -1
    # If it's a draw
    if check_draw(board):
        return 0
    # If the game is still ongoing
    return 0

### Question 5

In [20]:
def minimax(board, current_player, player_ai, player_opponent):
    # Check for terminal states
    if check_win(board, player_ai):
        return get_utility(board, player_ai, player_opponent)
    if check_win(board, player_opponent):
        return get_utility(board, player_ai, player_opponent)
    if check_draw(board):
        return get_utility(board, player_ai, player_opponent)

    # If current_player is the AI (maximizing player)
    if current_player == player_ai:
        max_eval = float('-inf')
        for next_board in possible_moves(board, current_player):
            eval = minimax(next_board, player_opponent, player_ai, player_opponent)
            max_eval = max(max_eval, eval)
        return max_eval
    # If current_player is the opponent (minimizing player)
    else:
        min_eval = float('inf')
        for next_board in possible_moves(board, current_player):
            eval = minimax(next_board, player_ai, player_ai, player_opponent)
            min_eval = min(min_eval, eval)
        return min_eval

In [21]:
def find_best_move(board, player_ai, player_opponent):
    best_move = None
    max_eval = float('-inf')

    # Iterate through all possible moves for the AI
    for next_board in possible_moves(board, player_ai):
        # Call minimax for the opponent's turn after the AI's move
        eval = minimax(next_board, player_opponent, player_ai, player_opponent)

        # Update max_eval and best_move if a better move is found
        if eval > max_eval:
            max_eval = eval
            best_move = next_board

    return best_move

#### Functions for Game Simulation

In [23]:
def display_board(board):
    for row in board:
        print(" | ".join(row))
        print("---------")

In [25]:
def play_game_ai_vs_ai():
    current_board = empty_state
    player_ai_X = 'X'
    player_ai_O = 'O'
    current_turn = player_ai_X # X starts first

    print("\n--- Starting Tic-Tac-Toe Game (AI vs AI) ---")
    display_board(current_board)

    while True:
        print(f"\n{current_turn}'s Turn")

        # AI makes a move
        # find_best_move returns the new board state after the optimal move
        new_board = find_best_move(current_board, current_turn, player_ai_X if current_turn == player_ai_O else player_ai_O)

        if new_board is None:
            print("No valid moves left. Something is wrong or it's a draw.")
            break

        current_board = new_board
        display_board(current_board)

        if check_win(current_board, current_turn):
            print(f"\n*** Player {current_turn} wins! ***")
            break
        elif check_draw(current_board):
            print("\n*** It's a Draw! ***")
            break

        # Switch turns
        current_turn = player_ai_O if current_turn == player_ai_X else player_ai_X

In [26]:
play_game_ai_vs_ai()

print("Game simulation complete.")


--- Starting Tic-Tac-Toe Game (AI vs AI) ---
  |   |  
---------
  |   |  
---------
  |   |  
---------

X's Turn
X |   |  
---------
  |   |  
---------
  |   |  
---------

O's Turn
X |   |  
---------
  | O |  
---------
  |   |  
---------

X's Turn
X | X |  
---------
  | O |  
---------
  |   |  
---------

O's Turn
X | X | O
---------
  | O |  
---------
  |   |  
---------

X's Turn
X | X | O
---------
  | O |  
---------
X |   |  
---------

O's Turn
X | X | O
---------
O | O |  
---------
X |   |  
---------

X's Turn
X | X | O
---------
O | O | X
---------
X |   |  
---------

O's Turn
X | X | O
---------
O | O | X
---------
X | O |  
---------

X's Turn
X | X | O
---------
O | O | X
---------
X | O | X
---------

*** It's a Draw! ***
Game simulation complete.
